# 02 — Train baseline detectors

Train normal-only detectors on clean benign runs and calibrate thresholds on clean benign validation runs.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

from robustedge.features import infer_feature_columns
from robustedge.models import default_detectors
from robustedge.calibration import QuantileCalibrator
from robustedge.pipeline import split_clean_benign_runs

OUTPUT_DIR = Path('../outputs/notebook_run')
MODEL_DIR = OUTPUT_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
features = pd.read_csv(OUTPUT_DIR / 'features.csv')
feature_cols = infer_feature_columns(features)
train_runs, val_runs = split_clean_benign_runs(features)

X_train = features[features.run_id.isin(train_runs)][feature_cols].to_numpy(float)
X_val = features[features.run_id.isin(val_runs)][feature_cols].to_numpy(float)

scaler = StandardScaler().fit(X_train)
joblib.dump(scaler, MODEL_DIR / 'scaler.joblib')
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)

thresholds = {}
for det in default_detectors(include_autoencoder=True):
    print('training', det.name)
    det.fit(X_train_s)
    scores = det.score(X_val_s)
    cal = QuantileCalibrator(quantile=0.995).fit(scores)
    thresholds[det.name] = cal.threshold_
    joblib.dump(det, MODEL_DIR / f'detector_{det.name}.joblib')

(OUTPUT_DIR / 'thresholds.json').write_text(json.dumps(thresholds, indent=2))
thresholds